# Notebook 01:  Data Quality Audit & Cleaning Pipeline

This notebook performs a structured data quality audit of the raw nested JSON dataset (`raw_credit_applications.json`).  

The objective is to:
- Examine structural characteristics of the dataset
- Identify data quality issues across key dimensions 
- Quantify the extent of each issue
- Implement reproducible remediation steps
- Produce a cleaned dataset suitable for downstream bias and governance analysis


This notebook is structured as follows:

1. Data Structure Overview
2. Data Quality Assessment
   - Completeness
   - Uniqueness
   - Validity
   - Consistency
   - Timeliness
   - Accuracy
3. MongoDB Governance Audit
4. Remediation Strategy & Cleaning Pipeline
5. Clean Dataset Summary

## 1. Data Structure and Basic Characteristics
Before performing quality checks, we examine the structural properties of the dataset to understand its format, schema, and potential structural risks. 

In [1]:
import pandas as pd
import numpy as np

df = pd.read_json("../data/raw_credit_applications.json")
df.head()

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData types:")
print(df.dtypes)

Shape: (502, 8)

Columns:
Index(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision',
       'processing_timestamp', 'loan_purpose', 'notes'],
      dtype='object')

Data types:
_id                     object
applicant_info          object
financials              object
spending_behavior       object
decision                object
processing_timestamp    object
loan_purpose            object
notes                   object
dtype: object


### Observations: 

- The dataset contains **502 credit applications** and **8 top-level fields**.
- The structure is **semi-structured (nested JSON)**, meaning several fields contain embedded objects and arrays:
  - `applicant_info` (nested object)
  - `financials` (nested object)
  - `spending_behavior` (array of objects)
  - `decision` (nested object)

- All columns are currently loaded as type `object`, indicating that nested fields have not yet been normalized.

### Structural Implications

Because the dataset is not flat:

- Data type inconsistencies may be hidden inside nested fields.
- Missing values inside nested objects may not be immediately visible.
- Array structures (e.g., `spending_behavior`) require special handling to avoid data loss.
- Schema inconsistencies may exist across records.

Therefore, normalization of nested structures is required before conducting a full data quality audit.

### Expected Schema (Data Contract)

To evaluate data quality consistently, we define expected structure and rules for key fields:

**Top-level**
- `_id`: unique application identifier (string)
- `processing_timestamp`: timestamp of processing (datetime, important for auditability)
- `loan_purpose`, `notes`: optional metadata fields

**Nested fields**
- `applicant_info`: personal identifiers and demographics (PII present: `ssn`, `email`)
- `financials`: numeric credit-risk features (e.g., income, credit history, debt-to-income)
- `decision`: approval outcome and dependent decision attributes
- `spending_behavior`: array of spending records (`category`, `amount`) — requires aggregation or explosion for validation

These expectations act as the reference baseline for completeness, validity, consistency, and timeliness checks.

# 2. Data Quality Assessment 

We assess the dataset using the six data quality dimensions defined in the course:

1. Completeness  
2. Uniqueness  
3. Validity  
4. Consistency  
5. Timeliness  
6. Accuracy  

## 2.1 Completeness

Completeness evaluates whether required data fields are present and populated.

Given the nested JSON structure, completeness must be assessed both at:
- Top-level fields
- Nested object fields (applicant_info, financials, decision)

In [3]:
df.isnull().sum()

_id                       0
applicant_info            0
financials                0
spending_behavior         0
decision                  0
processing_timestamp    440
loan_purpose            452
notes                   500
dtype: int64

Although nested objects (`applicant_info`, `financials`, `decision`) show 0 missing values at the top level, this does not guarantee completeness within their internal fields. Therefore, nested structures must be evaluated separately.

In [4]:
applicant_df = pd.json_normalize(df["applicant_info"])
financials_df = pd.json_normalize(df["financials"])
decision_df = pd.json_normalize(df["decision"])

In [47]:
print("Applicant missing:")
print((applicant_df.isnull() | (applicant_df == "")).sum())

print("\nFinancials missing:")
print((financials_df.isnull() | (financials_df == "")).sum())

print("\nDecision missing:")
print((decision_df.isnull() | (decision_df == "")).sum())

Applicant missing:
full_name        0
email            7
ssn              5
ip_address       5
gender           3
date_of_birth    5
zip_code         2
dtype: int64

Financials missing:
annual_income              5
credit_history_months      0
debt_to_income             0
savings_balance            0
annual_salary            497
dtype: int64

Decision missing:
loan_approved         0
rejection_reason    292
interest_rate       210
approved_amount     210
dtype: int64


The missingness pattern in decision fields is conditional:
- `interest_rate` and `approved_amount` are populated only when `loan_approved = True`.
- `rejection_reason` is populated only when `loan_approved = False`.

Therefore, this missingness reflects business logic rather than structural incompleteness.

In [48]:
total = len(df)

print("Top-level missing percentage:")
print((df.isnull().sum() / total * 100).round(2))

print("\nApplicant missing percentage:")
print(((applicant_df.isnull() | (applicant_df == "")).sum() / total * 100).round(2))

print("\nFinancials missing percentage:")
print(((financials_df.isnull() | (financials_df == "")).sum() / total * 100).round(2))

print("\nDecision missing percentage:")
print(((decision_df.isnull() | (decision_df == "")).sum() / total * 100).round(2))

Top-level missing percentage:
_id                      0.00
applicant_info           0.00
financials               0.00
spending_behavior        0.00
decision                 0.00
processing_timestamp    87.65
loan_purpose            90.04
notes                   99.60
dtype: float64

Applicant missing percentage:
full_name        0.00
email            1.39
ssn              1.00
ip_address       1.00
gender           0.60
date_of_birth    1.00
zip_code         0.40
dtype: float64

Financials missing percentage:
annual_income             1.0
credit_history_months     0.0
debt_to_income            0.0
savings_balance           0.0
annual_salary            99.0
dtype: float64

Decision missing percentage:
loan_approved        0.00
rejection_reason    58.17
interest_rate       41.83
approved_amount     41.83
dtype: float64


In [7]:
empty_spending = df["spending_behavior"].apply(
    lambda x: len(x) == 0 if isinstance(x, list) else True
)

print("Empty spending_behavior records:")
print(empty_spending.sum(), "(", round(empty_spending.mean()*100, 2), "% )")

Empty spending_behavior records:
0 ( 0.0 % )


No empty `spending_behavior` arrays were detected.  
All records contain at least one spending entry, indicating structural completeness for this field.

### Findings – Completeness

Top-level fields:
- `processing_timestamp` is missing in 87.65% of records.
- `loan_purpose` is missing in 90.04% of records.
- `notes` is missing in 99.60% of records.

Nested applicant fields:
- Minor missingness in `ssn` (1%) and `ip_address` (1%).
- Very low missingness in demographic attributes (≤ 0.2%).

Decision fields:
- Missingness in `interest_rate`, `approved_amount`, and `rejection_reason` is conditional based on loan approval status and reflects business logic rather than structural incompleteness.

Spending behavior:
- No empty arrays detected (0%).

Overall assessment:
Completeness issues are concentrated in optional metadata fields and audit-related timestamps rather than core financial risk variables. The high absence of `processing_timestamp` represents a governance and traceability risk and will be addressed in Section 3.

## 2.2 Uniqueness 
Uniqueness evaluates whether records that should be distinct are not duplicated.

In [8]:
total_rows = len(df)

# 1) Duplicate _id (beyond first occurrence)
duplicate_id_rows = df["_id"].duplicated().sum()

# 2) Duplicate SSN (ignoring missing values)
duplicate_ssn_rows = (
    applicant_df.loc[applicant_df["ssn"].notna(), "ssn"]
    .duplicated()
    .sum()
)

# 3) Duplicate Email (ignoring missing values)
duplicate_email_rows = (
    applicant_df.loc[applicant_df["email"].notna(), "email"]
    .duplicated()
    .sum()
)

# Distinct SSN values that appear more than once (Mongo-aligned metric)
distinct_duplicate_ssn_values = (
    applicant_df.loc[applicant_df["ssn"].notna(), "ssn"]
    .value_counts()
    .gt(1)
    .sum()
)


{
    "total_rows": total_rows,
    "duplicate_id_rows_beyond_first": int(duplicate_id_rows),
    "duplicate_ssn_rows_beyond_first": int(duplicate_ssn_rows),
    "duplicate_email_rows_beyond_first": int(duplicate_email_rows),
}

{'total_rows': 502,
 'duplicate_id_rows_beyond_first': 2,
 'duplicate_ssn_rows_beyond_first': 3,
 'duplicate_email_rows_beyond_first': 8}

In [9]:
df[df["_id"].duplicated(keep=False)][["_id", "notes"]].sort_values("_id")

,_id,notes
383,app_001,NaN
455,app_001,DUPLICATE_ENTRY_ERROR
8,app_042,NaN
354,app_042,RESUBMISSION


### Findings – Uniqueness

Record-level uniqueness (_id): The dataset contains 2 duplicate _id occurrences beyond the first, affecting the IDs app_001 and app_042. This indicates that the dataset is not strictly unique at the application record level.

Applicant identifier duplication: We identified duplication in applicant identifiers:

- 3 duplicate SSN occurrences beyond the first

- 8 duplicate email occurrences beyond the first

Missing values were excluded from this calculation.

Interpretation: Duplicate _id values represent repeated application records, while duplicate SSNs and emails may reflect either repeated submissions, identity reuse, or potential data entry inconsistencies. These cases require controlled remediation rather than automatic deletion.   

## 2.3 Validity

Validity evaluates whether data values fall within acceptable and logical ranges.

### 2.3.1  Financial Field Validation

In [10]:
fin_val = financials_df.copy()
fin_val["_id"] = df["_id"].values  # traceability

# Convert safely
fin_val["annual_income"] = pd.to_numeric(fin_val["annual_income"], errors="coerce")

# Validity checks
neg_income = (fin_val["annual_income"] < 0)
neg_savings = (fin_val["savings_balance"] < 0)
neg_credit_months = (fin_val["credit_history_months"] < 0)

print("Negative annual income:", int(neg_income.sum()))
print("Negative savings balance:", int(neg_savings.sum()))
print("Negative credit history months:", int(neg_credit_months.sum()))

# Evidence table (only invalid rows)
fin_val[neg_income | neg_savings | neg_credit_months][
    ["_id","annual_income","savings_balance","debt_to_income","credit_history_months"]
].sort_values("_id").head(20)

Negative annual income: 0
Negative savings balance: 1
Negative credit history months: 2


,_id,annual_income,savings_balance,debt_to_income,credit_history_months
137,app_043,131000.0,53098,0.06,-10
162,app_156,25000.0,13641,0.21,-3
159,app_290,104000.0,-5000,0.27,39


#### Findings – Financial Field Validation

Two financial validity issues were identified:

- 1 record contains a negative savings balance.

- 2 records contain negative credit history months.

Negative credit history months are logically impossible and indicate clear data entry or ingestion errors.

Although limited in volume, these issues are structurally important for downstream credit risk modelling and governance controls.

### 2.3.2 Decision Field Validation

Decision-level validity checks were performed to ensure that financial outcome variables fall within logical numeric ranges. Specifically, interest rates and approved amounts were tested for negative values.

In [11]:
dec_val = decision_df.copy()
dec_val["_id"] = df["_id"].values  # traceability

# Ensure numeric conversion (safe coercion)
dec_val["interest_rate"] = pd.to_numeric(dec_val["interest_rate"], errors="coerce")
dec_val["approved_amount"] = pd.to_numeric(dec_val["approved_amount"], errors="coerce")

# Validity checks
neg_interest = dec_val["interest_rate"] < 0
neg_amount = dec_val["approved_amount"] < 0

print("Negative interest rate:", int(neg_interest.sum()))
print("Negative approved amount:", int(neg_amount.sum()))

# Evidence table (only invalid rows)
evidence = dec_val[neg_interest | neg_amount][
    ["_id", "loan_approved", "interest_rate", "approved_amount", "rejection_reason"]
].sort_values("_id")

if evidence.empty:
    print("No invalid decision records found (no negative interest_rate or approved_amount).")
else:
    display(evidence.head(20))

Negative interest rate: 0
Negative approved amount: 0
No invalid decision records found (no negative interest_rate or approved_amount).


#### Findings - Decisions Field Validation

Decision-related monetary fields were validated after safe numeric coercion.

No negative values were identified in either `interest_rate` or `approved_amount`.

This indicates that decision-level financial outputs fall within expected logical numeric ranges and do not exhibit invalid negative monetary values.

No invalid decision records were detected.

### 2.3.3 Cross-field logical validation

Beyond individual field validation, logical consistency between decision attributes was assessed.

Certain fields are conditionally dependent on the loan approval status:

- If `loan_approved` is True, both `interest_rate` and `approved_amount` should be populated.
- If `loan_approved` is False, `rejection_reason` should be populated.
- Financial terms should not be present for rejected applications.

These checks ensure business-rule consistency within the decision data.

In [12]:
cross_val = decision_df.copy()
cross_val["_id"] = df["_id"].values  # traceability

approved_missing_interest = (cross_val["loan_approved"] == True) & (cross_val["interest_rate"].isnull())
approved_missing_amount = (cross_val["loan_approved"] == True) & (cross_val["approved_amount"].isnull())
rejected_missing_reason = (cross_val["loan_approved"] == False) & (cross_val["rejection_reason"].isnull())
rejected_has_interest = (cross_val["loan_approved"] == False) & (cross_val["interest_rate"].notnull())
rejected_has_amount = (cross_val["loan_approved"] == False) & (cross_val["approved_amount"].notnull())

print("Approved but missing interest rate:", int(approved_missing_interest.sum()))
print("Approved but missing approved amount:", int(approved_missing_amount.sum()))
print("Rejected but missing rejection reason:", int(rejected_missing_reason.sum()))
print("Rejected but interest rate present:", int(rejected_has_interest.sum()))
print("Rejected but approved amount present:", int(rejected_has_amount.sum()))

# Optional evidence table
violations = cross_val[
    approved_missing_interest |
    approved_missing_amount |
    rejected_missing_reason |
    rejected_has_interest |
    rejected_has_amount
][["_id", "loan_approved", "interest_rate", "approved_amount", "rejection_reason"]]

if violations.empty:
    print("No cross-field logical violations detected.")
else:
    display(violations.sort_values("_id").head(20))

Approved but missing interest rate: 0
Approved but missing approved amount: 0
Rejected but missing rejection reason: 0
Rejected but interest rate present: 0
Rejected but approved amount present: 0
No cross-field logical violations detected.


#### Findings – Cross-Field Logical Validation

Cross-field business rules were evaluated to ensure conditional consistency between loan approval status and dependent decision attributes.

No logical inconsistencies were identified.

All approved applications contain valid `interest_rate` and `approved_amount` values.  
All rejected applications contain appropriate `rejection_reason` values and no financial terms.

This indicates strong internal business-rule consistency within the decision data structure.

## 2.4 Consistency 

Consistency evaluates whether data representations are uniform across records and aligned with expected schema definitions.

### 2.4.1 Schema Consistency – Financial Fields

In [13]:
fin_cons = financials_df.copy()
fin_cons["_id"] = df["_id"].values  # traceability

# Safe numeric conversion (audit only)
fin_cons["annual_income"] = pd.to_numeric(fin_cons.get("annual_income"), errors="coerce")
fin_cons["annual_salary"] = pd.to_numeric(fin_cons.get("annual_salary"), errors="coerce")

has_income = fin_cons["annual_income"].notna()
has_salary = fin_cons["annual_salary"].notna()

schema_patterns = pd.DataFrame({
    "pattern": ["only_annual_income", "only_annual_salary", "both_present", "neither_present"],
    "count": [
        int((has_income & ~has_salary).sum()),
        int((~has_income & has_salary).sum()),
        int((has_income & has_salary).sum()),
        int((~has_income & ~has_salary).sum()),
    ]
})

schema_patterns["pct"] = (schema_patterns["count"] / len(fin_cons) * 100).round(2)
schema_patterns

,pattern,count,pct
0,only_annual_income,497,99.0
1,only_annual_salary,5,1.0
2,both_present,0,0.0
3,neither_present,0,0.0


In [14]:
# Evidence: show examples for each pattern
display(fin_cons.loc[has_income & ~has_salary, ["_id", "annual_income", "annual_salary"]].head(5))
display(fin_cons.loc[~has_income & has_salary, ["_id", "annual_income", "annual_salary"]].head(5))
display(fin_cons.loc[has_income & has_salary, ["_id", "annual_income", "annual_salary"]].head(5))
display(fin_cons.loc[~has_income & ~has_salary, ["_id", "annual_income", "annual_salary"]].head(5))

# True mismatches only where BOTH fields are present
both = fin_cons.loc[has_income & has_salary, ["_id", "annual_income", "annual_salary"]].copy()

mismatch_mask = both["annual_income"] != both["annual_salary"]
mismatch_count = int(mismatch_mask.sum())

print("True mismatches (both present, values differ):", mismatch_count)

if mismatch_count > 0:
    display(both.loc[mismatch_mask].sort_values("_id").head(20))
else:
    print("No mismatches where both fields are present.")

,_id,annual_income,annual_salary
0,app_200,73000.0,NaN
1,app_037,78000.0,NaN
2,app_215,61000.0,NaN
3,app_024,103000.0,NaN
4,app_184,57000.0,NaN


,_id,annual_income,annual_salary
76,app_436,NaN,45000.0
94,app_421,NaN,46000.0
99,app_479,NaN,94000.0
141,app_463,NaN,86000.0
149,app_449,NaN,75000.0


,_id,annual_income,annual_salary


,_id,annual_income,annual_salary


True mismatches (both present, values differ): 0
No mismatches where both fields are present.


#### Findings – Schema Consistency (Financial Fields)

The financial schema shows structural fragmentation between the fields `annual_income` and `annual_salary`, which represent the same business concept.

Distribution of usage patterns:

- Only `annual_income` populated: 497 records (99%)
- Only `annual_salary` populated: 5 records (1%)
- Both fields populated: 0 records (0%)
- Neither field populated: 0 records (0%)

This indicates inconsistent schema usage, where the same attribute is stored across multiple columns instead of being standardized.

No records contain both fields simultaneously, therefore no direct value mismatches were identified. The issue is structural rather than conflicting data.

**Implication:** This schema fragmentation increases downstream processing complexity and creates risk of inconsistent feature engineering, as the same concept must be reconciled across multiple attributes.

Remediation will consolidate these attributes into a single standardized income field in Section 3.

Suggestion: This inconsistency should be resolved through schema consolidation and standardization into a single authoritative field.

### 2.4.2 Representation Consistency – Decision Fields

In [15]:
dec_cons = decision_df.copy()

print("Data type of loan_approved:")
print(dec_cons["loan_approved"].dtype)

print("\nUnique values in loan_approved:")
print(dec_cons["loan_approved"].unique())

Data type of loan_approved:
bool

Unique values in loan_approved:
[False  True]


In [16]:
print("\nUnique values in rejection_reason:")
print(dec_cons["rejection_reason"].dropna().unique())


Unique values in rejection_reason:
['algorithm_risk_score' 'insufficient_credit_history' 'high_dti_ratio'
 'low_income']


In [17]:
print("\nData types:")
print(dec_cons[["interest_rate", "approved_amount"]].dtypes)


Data types:
interest_rate      float64
approved_amount    float64
dtype: object


#### Findings – Representation Consistency (Decision Fields)

The `loan_approved` field is consistently encoded as a boolean variable (`True`/`False`) across all records, with no mixed string representations detected.

The `rejection_reason` field shows standardized categorical values, consistently formatted in lowercase without structural variations.

Monetary fields (`interest_rate`, `approved_amount`) are consistently represented as numeric (`float64`) types across all records.

Overall, decision-level attributes demonstrate strong representational consistency with no detected encoding inconsistencies.

### 2.4.3 Representation Consistency – Applicant Gender

In [18]:
print("Unique raw gender values:")
print(applicant_df["gender"].astype("string").str.strip().unique())

print("\nFrequency distribution:")
print(applicant_df["gender"].astype("string").str.strip().value_counts(dropna=False))

Unique raw gender values:
<StringArray>
['Male', 'M', 'F', 'Female', '', <NA>]
Length: 6, dtype: string

Frequency distribution:
gender
Male      195
Female    193
F          58
M          53
            2
<NA>        1
Name: count, dtype: Int64


In [19]:
inconsistent_pct = (58 + 53) / len(applicant_df) * 100
print(f"Inconsistent abbreviated encodings represent {round(inconsistent_pct,2)}% of records.")

Inconsistent abbreviated encodings represent 22.11% of records.


#### Findings – Representation Consistency (Applicant Gender)

The `gender` attribute exhibits representational inconsistency.

The following encodings are used interchangeably for the same categories:
- "Male" and "M"
- "Female" and "F"

Additionally:
- 2 records contain empty string values ("")
- 1 record contains a null (NA) value

Abbreviated encodings ("M", "F") represent 22.11% of records, indicating significant categorical fragmentation.

While semantically equivalent, these variations create separate categories at the data level, which can distort demographic reporting, fairness metrics, and downstream analytical models.

**Implication:** Categorical standardization is required prior to analytical use to ensure consistent aggregation and bias monitoring.

### 2.4 Representation Consistency – Applicant Date of Birth

In [20]:
dob_val = applicant_df.copy()
dob_val["_id"] = df["_id"].values  # traceability

# Clean raw string
dob_raw = dob_val["date_of_birth"].astype("string").str.strip()
dob_val["dob_raw"] = dob_raw

# Identify patterns (very simple but effective)
is_iso = dob_raw.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)          # YYYY-MM-DD
is_pt  = dob_raw.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)          # DD/MM/YYYY
is_empty = (dob_raw == "") | dob_raw.isna()

dob_val["dob_format"] = np.select(
    [is_iso, is_pt, is_empty],
    ["YYYY-MM-DD", "DD/MM/YYYY", "missing"],
    default="other/invalid"
)

# Summary counts + %
summary = (
    dob_val["dob_format"]
    .value_counts(dropna=False)
    .to_frame("count")
    .assign(pct=lambda x: (x["count"] / len(dob_val) * 100).round(2))
)

summary

,count,pct
dob_format,,
YYYY-MM-DD,340,67.73
DD/MM/YYYY,101,20.12
other/invalid,56,11.16
missing,5,1.00


#### Findings – Representation Consistency (Applicant Date of Birth)

The date_of_birth field exhibits significant representation inconsistency across records.

- 67.73% of values follow ISO format (YYYY-MM-DD).

- 20.12% follow DD/MM/YYYY.

- 11.16% are classified as other/invalid formats.

- 1.00% are missing.

The coexistence of multiple date formats introduces parsing ambiguity (particularly day–month inversion risk) and may lead to incorrect age derivation or downstream analytical errors.

Standardization into a single canonical format (ISO YYYY-MM-DD) is required to ensure consistent temporal representation and reliable demographic analysis.

### Overall Assessment – Consistency

Consistency analysis revealed mixed results across different domains:

- Financial fields exhibit structural schema fragmentation, with two separate attributes (annual_income and annual_salary) representing the same business concept.

- Decision fields demonstrate strong representational consistency, with uniform boolean and categorical encodings.

- Applicant gender shows representational inconsistency, with abbreviated and full-text encodings used interchangeably in 22.11% of records.

- Applicant date_of_birth presents format inconsistency, with multiple date representations (YYYY-MM-DD and DD/MM/YYYY) and additional invalid or non-standard formats.

Overall, consistency issues are primarily structural and representational rather than numerical.

While these inconsistencies do not invalidate the dataset, they increase preprocessing complexity and may introduce parsing ambiguity, fragmented categorical groupings, and potential analytical bias in downstream processes.

Standardization measures will be implemented in Section 3 to harmonize categorical encodings, consolidate fragmented schema elements, and normalize date representations.

## 2.5 Timeliness

Timeliness evaluates whether data is recorded and updated within an acceptable time frame 

In [21]:
# Convert safely to datetime
time_val = df.copy()
time_val["processing_timestamp"] = pd.to_datetime(
    time_val["processing_timestamp"], errors="coerce"
)

total_rows = len(time_val)
missing_ts = time_val["processing_timestamp"].isna().sum()

print("Missing processing_timestamp:", missing_ts)
print("Missing %:", round(missing_ts / total_rows * 100, 2))

Missing processing_timestamp: 440
Missing %: 87.65


In [22]:
print("Timestamp range:")
print("Min:", time_val["processing_timestamp"].min())
print("Max:", time_val["processing_timestamp"].max())

Timestamp range:
Min: 2024-01-15 00:00:00+00:00
Max: 2027-01-20 00:00:00+00:00


In [23]:
from datetime import datetime
import pandas as pd

current_year = 2026  # estamos em 2026

future_records = time_val[
    time_val["processing_timestamp"].dt.year > current_year
].shape[0]

print("Records with future timestamps:", future_records)

Records with future timestamps: 1


#### Findings – Timeliness

The dataset contains a single temporal attribute: processing_timestamp.

A significant limitation was identified: 440 records (87.65%) lack a processing timestamp. This severely restricts the ability to evaluate data freshness, processing latency, update recency, and temporal audit traceability.

For records where timestamps are present, values fall within a range between January 2024 and January 2027. However, one record contains a timestamp in 2027, which is in the future relative to the current system year (2026).

This future-dated record suggests a potential system clock inconsistency, test data contamination, or ingestion timing anomaly, and should be reviewed under data governance controls.

## 2.6 Accuracy

Accuracy evaluates whether data correctly represents real-world values.

In the absence of external reference datasets, accuracy was approximated through internal plausibility and logical consistency checks.

In [24]:
print("Income summary statistics:")
print(financials_df["annual_income"].describe())

Income summary statistics:
count       497
unique      132
top       79000
freq         11
Name: annual_income, dtype: int64


In [25]:
print("Savings summary statistics:")
print(financials_df["savings_balance"].describe())

Savings summary statistics:
count      502.000000
mean     29493.503984
std      16775.309756
min      -5000.000000
25%      17258.250000
50%      27385.500000
75%      38251.500000
max      88078.000000
Name: savings_balance, dtype: float64


In [26]:
print("DTI summary:")
print(financials_df["debt_to_income"].describe())

DTI summary:
count    502.000000
mean       0.246195
std        0.136296
min        0.050000
25%        0.150000
50%        0.230000
75%        0.350000
max        1.850000
Name: debt_to_income, dtype: float64


### Findings – Accuracy

Accuracy was approximated through internal plausibility analysis due to the absence of an external ground-truth reference dataset.

**Annual Income**
Annual income values range between 0 and 171,000, with a mean of approximately 82,705 and a median of 81,000. The distribution appears statistically plausible and does not exhibit extreme outliers or unrealistic magnitudes.

**Savings Balance**
Savings balances range between -5,000 and 88,078. With the exception of a small number of negative values (previously identified under validity checks), values fall within realistic financial ranges.

**Debt-to-Income Ratio**
Debt-to-income ratios show a mean of 0.246 (24.6%) and a median of 0.23, which aligns with realistic borrower profiles. The maximum value of 1.85 represents an extreme case already captured under validity checks.

Overall, no systemic numerical distortions were detected. Accuracy concerns are limited to isolated anomalies previously identified in the validity assessment.

# 3. MongoDB Governance Audit (raw layer validation)

To complement the pandas-based profiling, MongoDB aggregation pipelines were used to validate governance risks directly at the raw data layer.

The raw JSON dataset originally contained 502 records.
During MongoDB import, 2 documents were rejected due to _id uniqueness violations (duplicate primary keys).
As a result, 500 records were retained in the MongoDB raw layer.

### 3.1 Uniqueness – Duplicate SSN Detection

In [27]:
from pymongo import MongoClient
import pandas as pd

# Connect
client = MongoClient("mongodb://localhost:27017/")
db = client["credit_db"]
collection = db["credit_applications"]

# 3.1 Uniqueness — Duplicate SSN Detection
pipeline = [
    {"$match": {"applicant_info.ssn": {"$exists": True, "$ne": None, "$ne": ""}}},
    {"$group": {"_id": "$applicant_info.ssn", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 1}}},
    {"$sort": {"count": -1}}
]

dup_ssn = list(collection.aggregate(pipeline))

print("Duplicate SSNs found:", len(dup_ssn))
if dup_ssn:
    display(pd.DataFrame(dup_ssn).rename(columns={"_id": "ssn"}).head(10))

Duplicate SSNs found: 0


### 3.2 Consistency - Gender Encoding Distribution

In [28]:
pipeline = [
    {"$group": {"_id": "$applicant_info.gender", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]

gender_counts = list(collection.aggregate(pipeline))

print("Distinct gender values (raw):", len(gender_counts))
if gender_counts:
    display(pd.DataFrame(gender_counts).rename(columns={"_id": "gender"}))

Distinct gender values (raw): 0


### 3.3 Timeliness & Completeness – Processing Timestamp Validation

In [29]:
pipeline = [
    {
        "$match": {
            "processing_timestamp": {
                "$gt": "2026-12-31T23:59:59Z"
            }
        }
    },
    {"$count": "future_records"}
]

future_records = list(collection.aggregate(pipeline))
future_count = future_records[0]["future_records"] if future_records else 0

print("Future processing_timestamp records:", future_count)

Future processing_timestamp records: 0


In [30]:
pipeline_missing = [
    {
        "$match": {
            "$or": [
                {"processing_timestamp": {"$exists": False}},
                {"processing_timestamp": None},
                {"processing_timestamp": ""}
            ]
        }
    },
    {"$count": "missing_processing_timestamp"}
]

missing = list(collection.aggregate(pipeline_missing))
missing_count = missing[0]["missing_processing_timestamp"] if missing else 0

print("Missing processing_timestamp records:", missing_count)

Missing processing_timestamp records: 0


### 3.4 Validity – Financial Range Violations

In [31]:
pipeline_invalid_financials = [
    {
        "$match": {
            "$or": [
                {"financials.annual_income": {"$lt": 0}},
                {"financials.savings_balance": {"$lt": 0}},
                {"financials.debt_to_income_ratio": {"$gt": 1}}
            ]
        }
    },
    {"$count": "invalid_financial_records"}
]

invalid_financials = list(collection.aggregate(pipeline_invalid_financials))
invalid_count = invalid_financials[0]["invalid_financial_records"] if invalid_financials else 0

print("Invalid financial records (range violations):", invalid_count)

Invalid financial records (range violations): 0


The MongoDB raw-layer audit confirmed the main governance risks identified during pandas profiling, ensuring cross-layer validation consistency before remediation.

## 4. Remediation Strategy & Cleaning Pipeline

Following the identification and quantification of data quality issues, this section demonstrates concrete remediation steps implemented to improve structural integrity, logical validity, and analytical usability of the dataset.

All transformations are reproducible and form a structured data cleaning pipeline.

In [32]:
df_clean = df.copy()

We create a separate working dataset (`df_clean`) to apply remediation steps without altering the original raw dataset used in Section 2.

### 4.1 Schema Standardization (Income Fields)

In [33]:
financials_clean = pd.json_normalize(df_clean["financials"])

financials_clean["annual_income_clean"] = (
    financials_clean["annual_income"]
    .fillna(financials_clean["annual_salary"])
)

df_clean["financials"] = financials_clean.to_dict(orient="records")

To resolve schema fragmentation between `annual_income` and `annual_salary`, a consolidated field `annual_income_clean` was created within the nested `financials` structure.

The consolidation rule applied was:
- Use `annual_income` when available.
- Otherwise, fallback to `annual_salary`.

The original fields were preserved to maintain auditability and traceability of the transformation process.

This approach ensures structural standardization while retaining transparency of the original data representation.

### 4.2 Gender Standardization 

To resolve the representation inconsistency in the `gender` field, values will be standardized into a controlled vocabulary:

- `male`: maps from {"Male", "M"}
- `female`: maps from {"Female", "F"}

Empty strings ("") and null values will be converted to missing (NA) to avoid introducing incorrect labels.

This standardization ensures consistent grouping for demographic analysis and prevents fragmented categories in downstream fairness metrics. The transformation was applied within `df_clean` while preserving the original raw dataset.

In [34]:
applicant_clean = pd.json_normalize(df_clean["applicant_info"])

def normalize_gender(x):
    if pd.isna(x):
        return pd.NA
    
    s = str(x).strip()
    if s == "":
        return pd.NA
    
    s_lower = s.lower()
    if s_lower in ["m", "male"]:
        return "male"
    if s_lower in ["f", "female"]:
        return "female"
    
    return pd.NA

# Create clean column
applicant_clean["gender_clean"] = applicant_clean["gender"].apply(normalize_gender)

# Put back into df_clean (nested structure preserved)
df_clean["applicant_info"] = applicant_clean.to_dict(orient="records")

# Validation
print("After cleaning:")
print(applicant_clean["gender_clean"].value_counts(dropna=False))

After cleaning:
gender_clean
female    251
male      248
<NA>        3
Name: count, dtype: int64


### 4.3 Date Standardization – date_of_birth

To resolve representation inconsistencies identified in Section 2, the date_of_birth field was standardized into a canonical ISO format (YYYY-MM-DD).
Explicit parsing rules were applied to avoid day–month inversion errors, and values that could not be reliably interpreted were conservatively set to missing (NA).

In [35]:
applicant_clean = pd.json_normalize(df_clean["applicant_info"]).copy()

dob_col = None
for c in ["date_of_birth", "date-of-birth"]:
    if c in applicant_clean.columns:
        dob_col = c
        break

if dob_col is None:
    raise KeyError("Não encontrei 'date_of_birth' nem 'date-of-birth' dentro de applicant_info.")

# Strip whitespace and fix any JSON-escaped slashes (e.g., "10\/08\/1980" -> "10/08/1980")
dob_fixed = applicant_clean[dob_col].astype("string").str.strip().str.replace(r'\\/', '/', regex=True)

# Use pandas' flexible parser to automatically handle US, EU, and single-digit formats
applicant_clean["date_of_birth_clean"] = pd.to_datetime(dob_fixed, format='mixed', errors="coerce").dt.strftime("%Y-%m-%d")

# Reintegrate the cleaned data
df_clean["applicant_info"] = applicant_clean.to_dict(orient="records")

# (Optional) Map the clean dates back directly to the flat column your age function uses
df_clean["applicant_info.date_of_birth_clean"] = applicant_clean["date_of_birth_clean"]

total = len(applicant_clean)
parsed = applicant_clean["date_of_birth_clean"].notna().sum()
missing = applicant_clean["date_of_birth_clean"].isna().sum()

print("After standardization:")
print("Parsed:", int(parsed))
print("Missing (including invalid):", int(missing))
print("Parsed %:", round(parsed / total * 100, 2))

After standardization:
Parsed: 497
Missing (including invalid): 5
Parsed %: 99.0


### 4.4 Invalid Value Handling

In [36]:
# Normalize nested financials from df_clean
financials_clean = pd.json_normalize(df_clean["financials"])

# Negative credit history months → set to NA
financials_clean.loc[
    financials_clean["credit_history_months"] < 0,
    "credit_history_months"
] = pd.NA

# Negative savings → set to NA
financials_clean.loc[
    financials_clean["savings_balance"] < 0,
    "savings_balance"
] = pd.NA

# Put cleaned financials back into df_clean
df_clean["financials"] = financials_clean.to_dict(orient="records")

# Quick validation check
print("Remaining negative credit history:",
      (financials_clean["credit_history_months"] < 0).sum())

print("Remaining negative savings:",
      (financials_clean["savings_balance"] < 0).sum())

Remaining negative credit history: 0
Remaining negative savings: 0


Logical validity violations identified in Section 2 were addressed conservatively by converting strictly invalid values to missing (NA), rather than attempting speculative corrections.

The following remediation rules were applied:

- Negative credit_history_months → set to NA

- Negative savings_balance → set to NA

### 4.5 Duplicate Handling

In [37]:

# 1) _id deduplication (record-level). Keep first, quarantine the removed ones for audit.
dup_id_mask = df_clean["_id"].duplicated(keep="first")

dq_quarantine_id_duplicates = df_clean.loc[dup_id_mask].copy()
df_clean = df_clean.loc[~dup_id_mask].copy()

print("Rows after _id dedup:", len(df_clean))
print("Quarantined _id-duplicate rows:", len(dq_quarantine_id_duplicates))

# 2) Recompute applicant info from df_clean for alignment
applicant_clean = pd.json_normalize(df_clean["applicant_info"])

# 3) Duplicate flags (SSN/email) excluding missing values
dq_flag_duplicate_note = df_clean["notes"].astype("string").str.contains("DUPLICATE_ENTRY_ERROR", na=False)

ssn_present = applicant_clean["ssn"].notna()
email_present = applicant_clean["email"].notna()

dq_flag_duplicate_ssn = ssn_present & applicant_clean["ssn"].duplicated(keep=False)
dq_flag_duplicate_email = email_present & applicant_clean["email"].duplicated(keep=False)

# 4) Attach flags back to df_clean
df_clean["dq_flag_duplicate_note"] = dq_flag_duplicate_note.values
df_clean["dq_flag_duplicate_ssn"] = dq_flag_duplicate_ssn.values
df_clean["dq_flag_duplicate_email"] = dq_flag_duplicate_email.values

df_clean["dq_duplicate_any"] = (
    df_clean["dq_flag_duplicate_note"] |
    df_clean["dq_flag_duplicate_ssn"] |
    df_clean["dq_flag_duplicate_email"]
)

# Governance action: quarantine for review rather than auto-delete
df_clean["dq_action"] = np.where(df_clean["dq_duplicate_any"], "quarantine_review", "ok")

print("\ndq_action counts:")
print(df_clean["dq_action"].value_counts())

print("\nTop flag patterns:")
print(
    df_clean[["dq_flag_duplicate_note", "dq_flag_duplicate_ssn", "dq_flag_duplicate_email", "dq_action"]]
    .value_counts()
    .head(10)
)

Rows after _id dedup: 500
Quarantined _id-duplicate rows: 2

dq_action counts:
dq_action
ok                   489
quarantine_review     11
Name: count, dtype: int64

Top flag patterns:
dq_flag_duplicate_note  dq_flag_duplicate_ssn  dq_flag_duplicate_email  dq_action        
False                   False                  False                    ok                   489
                                               True                     quarantine_review      7
                        True                   False                    quarantine_review      4
Name: count, dtype: int64


In [38]:
dq_quarantine_id_duplicates[["_id", "notes"]].sort_values("_id")

,_id,notes
455,app_001,DUPLICATE_ENTRY_ERROR
354,app_042,RESUBMISSION


Duplicate management follows a governance-safe approach.

- **Record-level duplicates (`_id`)** were deduplicated by keeping the first occurrence. This reduced the dataset from **502 to 500 records**, and **2 duplicate rows** were quarantined for auditability (`app_001`, `app_042`).
- **Applicant-level duplicates (SSN/email)** were not automatically deleted because they may represent legitimate resubmissions or identifier reuse. Instead, affected records were flagged and assigned a governance action (`dq_action = quarantine_review`).

After applying duplicate governance rules:
- **489 records** were marked as `ok`
- **11 records** were marked as `quarantine_review` (7 driven by email duplication, 4 driven by SSN duplication)

This strategy improves uniqueness while preserving traceability and minimizing the risk of removing valid real-world records.

### 4.6 Timestamp Normalization

In [39]:
df_clean["processing_timestamp"] = pd.to_datetime(
    df_clean["processing_timestamp"],
    errors="coerce",
    utc=True
)

print("Timestamp dtype:", df_clean["processing_timestamp"].dtype)
print("Missing timestamps after normalization:",
      df_clean["processing_timestamp"].isna().sum())


reference_date = pd.Timestamp("2026-12-31 23:59:59", tz="UTC")

is_future = (
    df_clean["processing_timestamp"].notna() &
    (df_clean["processing_timestamp"] > reference_date)
)

df_clean["dq_flag_future_processing_ts"] = is_future

future_count = is_future.sum()

print("Future timestamps detected:", int(future_count))

Timestamp dtype: datetime64[ns, UTC]
Missing timestamps after normalization: 438
Future timestamps detected: 1


The processing_timestamp field was standardized to a timezone-aware datetime format (UTC) using safe coercion.

This step ensures:

- Consistent temporal representation

- Elimination of timezone ambiguity (tz-naive vs tz-aware inconsistencies)

- Prevention of downstream comparison errors

Invalid or unparsable timestamp values were converted to missing (NaT) to preserve data integrity.
No artificial imputation was performed due to the high proportion of missing values and the absence of reliable reference temporal attributes.

Additional timeliness validation was applied using a fixed project reference date (2026-12-31 UTC).
Records with processing timestamps beyond this boundary were flagged for review rather than automatically corrected, ensuring reproducibility and preserving audit traceability.

## 5. Clean Dataset Summary

In [40]:
print("Final dataset shape:", df_clean.shape)

Final dataset shape: (500, 15)


In [41]:
financials_clean = pd.json_normalize(df_clean["financials"])

print("Missing annual_income_clean:",
      financials_clean["annual_income_clean"].isna().sum())

Missing annual_income_clean: 0


In [42]:
applicant_clean = pd.json_normalize(df_clean["applicant_info"])

print(applicant_clean["gender_clean"].value_counts(dropna=False))

gender_clean
female    251
male      247
None        2
Name: count, dtype: int64


In [43]:
applicant_clean = pd.json_normalize(df_clean["applicant_info"])

print("Missing standardized date_of_birth_clean:",
      applicant_clean["date_of_birth_clean"].isna().sum())

print("Valid date_of_birth_clean records:",
      applicant_clean["date_of_birth_clean"].notna().sum())

Missing standardized date_of_birth_clean: 4
Valid date_of_birth_clean records: 496


In [44]:
print("Remaining negative credit history:",
      (financials_clean["credit_history_months"] < 0).sum())

print("Remaining negative savings:",
      (financials_clean["savings_balance"] < 0).sum())

Remaining negative credit history: 0
Remaining negative savings: 0


In [45]:
print(df_clean["dq_action"].value_counts())
# Saves the cleaned dataset for bias analysis and future use
df_clean.to_json('../data/cleaned_dataset.json', index=False)

dq_action
ok                   489
quarantine_review     11
Name: count, dtype: int64


After applying remediation rules, the dataset reflects improved structural integrity and standardized representations.

Key outcomes:

- Record count reduced from 502 to 500 after record-level deduplication.
- Income schema fragmentation resolved through consolidation into a single authoritative field (`annual_income_clean`).
- Gender values standardized to a controlled vocabulary (`male`, `female`), eliminating abbreviated encodings.
- Date of birth values standardized to ISO format (`YYYY-MM-DD`), resolving mixed format representations and isolating invalid entries.
- Logical validity violations (negative credit history months and negative savings balances) converted to missing values (NA) using a conservative correction approach.
- Duplicate governance flags applied without destructive removal of potentially legitimate resubmissions.
- Processing timestamps normalized to consistent UTC datetime format.

The cleaned dataset (`df_clean`) is now structurally standardized, representation-consistent, audit-traceable, and suitable for downstream analytical use.